# 2. Basic Statistics & Summary <a id="2-basic-statistics" name="2-basic-statistics"></a>

In [ ]:
# Descriptive statistics
print("DESCRIPTIVE STATISTICS — Numerical columns:")
display(df_raw.describe().round(2))

DESCRIPTIVE STATISTICS — Numerical columns:


,id,runtime,budget,revenue,popularity,vote_average,vote_count
count,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00
mean,288807.82,108.55,27972659.11,91988515.29,7.51,6.83,2681.80
std,413194.38,29.73,46898167.80,192359913.48,15.56,1.04,4023.13
min,11.00,0.00,0.00,0.00,0.02,0.00,0.00
25%,9315.50,94.00,0.00,0.00,3.99,6.38,376.00
50%,38588.50,106.00,7000000.00,17387297.50,5.32,6.93,1198.50
75%,466580.50,121.00,35000000.00,100163395.00,7.43,7.46,3263.25
max,1659087.00,960.00,489900000.00,2923706026.00,683.14,10.00,35532.00


**ID :**

mean : 288,807.82 & max : 1,659,087. TMDB IDs are not sequential, it is a global database that grows over time !

**Runtime :**

The average duration is 108 min. We have movies with a minimum duration = 0, indicating missing data, and a maximum duration of 960 min (16 hours), which is an outlier ("experimental film or data entry error ?").

**Budget :**

25% = $0. At least 25% of the movies have a budget of $0, these are not actual zero budgets, but missing data. We have a very high std = $46.8M, showing an enormous dispersion between blockbusters and independent films.

**Revenue :**

The same issue as budget with a lot of missing data (min = $0, max = $2.92B, likely Avatar or Avengers). The gap between the mean ($91M) and the median ($17M) is huge >> highly skewed distribution pulled upward by blockbusters.

**Popularity :**

TMDB popularity score: a max of 683 while the average is 7.5 >> massive outliers, std: 15.56 > mean: 7.51 >> extremely skewed distribution where a few ultra-popular movies pull the average upward.

**Vote_Average :**

"Average Rating" : low std : 1.04, making it the healthiest distribution in the dataset. Ratings are concentrated between 6 and 8. We also notice a min of 0 (movies with no votes) > needs to be filtered out for recommendation.

**Vote_count (Number of votes) :**

min = 0 for movies without any votes, making their ratings unusable ! A movie with a vote_average of 8 but a vote_count of 2 is not reliable, so we must set a minimum threshold (vote_count > 100) for recommendation.

In [ ]:
# Categorical summaries
print(" CATEGORICAL SUMMARIES:")
for col in ["status","original_language","adult"]:
    vc = df_raw[col].value_counts()
    print(f"\n  {col}:")
    print(vc.head(10).to_string()) # to_strings : convert


 CATEGORICAL SUMMARIES:

  status:
status
Released           5477
Post Production      16
In Production         5
Planned               2

  original_language:
original_language
en    4325
ja     266
fr     179
ko     115
it     111
es      97
cn      69
zh      67
de      50
ru      29

  adult:
adult
False    5500


**Status :**

5,477 released movies, 23 movies not yet released ("post_production, in production, and planned"). Therefore, we filter exclusively for status == "Released" for recommendation, as an unreleased movie lacks reliable ratings.

**Original_language :**

Significant bias (English-biased dataset): 4,325 movies are in English, followed by 10 other languages in the top 10. This information is very useful, as it allows filtering by language in the Streamlit interface.

**Adult :**

No adult movies in the dataset. We can drop this column from the dataset > useless column.   

In [ ]:
# KPI :
print("="*55)
print("  KPI clés — Raw TMDB Dataset")
print("="*55)
print(f"  Total films:           {len(df_raw):,}")
print(f"  Unique directors:      {df_raw['director'].nunique():,}")
print(f"  Year range:            {pd.to_datetime(df_raw['release_date'],errors='coerce').dt.year.min():.0f} – {pd.to_datetime(df_raw['release_date'],errors='coerce').dt.year.max():.0f}")
print(f"  Avg rating:            {df_raw['vote_average'].mean():.2f} / 10")
print(f"  Avg runtime:           {df_raw['runtime'].mean():.0f} min")
print(f"  Films with budget:     {(df_raw['budget']>0).sum():,}  ({(df_raw['budget']>0).mean()*100:.1f}%)")
print(f"  Films with revenue:    {(df_raw['revenue']>0).sum():,}  ({(df_raw['revenue']>0).mean()*100:.1f}%)")
print(f"  Films with poster:     {df_raw['poster_path'].notna().sum():,}  ({df_raw['poster_path'].notna().mean()*100:.1f}%)")
print(f"  Languages present:     {df_raw['original_language'].nunique():,}")


  KPI clés — Raw TMDB Dataset
  Total films:           5,500
  Unique directors:      2,864
  Year range:            1902 – 2028
  Avg rating:            6.83 / 10
  Avg runtime:           109 min
  Films with budget:     3,670  (66.7%)
  Films with revenue:    3,940  (71.6%)
  Films with poster:     5,498  (100.0%)
  Languages present:     44


**Total movies :**

We have 5,500 total movies, which is ideal for a recommendation engine.

**Unique_directors :**

2,864 >> 5,500 movies / 2,864 directors ≈ on average 1.9 movies per director > good diversity, the movies are not dominated by a few directors >> some directors have many movies, making this column useful for recommendation.

**Year Range :**

1902–2028 : good coverage. 2028 movies are planned or in production, which we will filter out for recommendation.

**AVG Rating :**

6.83/10: consistent with TMDB. The global average is 6.57; TMDB users tend to rate rather positively (selection bias: people mostly rate movies they chose to watch and thus already appreciate).

**Avg_runtime :**

109 min: consistent, as a movie typically lasts between 90 and 120 minutes.

**Movies with budget :**

3,670 ($) (66.7%) : 1 out of 3 movies does not have budget data reported. Movies without budget often represent independent or foreign productions. For financial analysis, we will work strictly on movies with a specified budget.

**Movies with Revenue :**

3,940 ($) (71.6%) >> better than budget: some movies have a known budget but no declared revenue. Important for calculating ROI.

**Movies with poster :**

5,498 (100%): only 2 movies without a poster. Important for the Streamlit app, as we can display poster images for almost every movie.

**Languages present :**

44 >> great linguistic diversity across the 5,500 movies, useful for language filtering in the Streamlit interface.